# cuda-dl-kernels: optimizing deep learning operators on a T4

**Kaggle setup:** Settings -> Accelerator -> **GPU T4 x2** (P100 has no tensor cores). Internet: On. Then Run All (~10 min).

| Track | Stages | Baseline |
|---|---|---|
| FP32 SGEMM | naive -> coalesced -> shared-mem tiling -> 2D register blocking -> float4 vectorized | cuBLAS FP32 |
| FP16 tensor cores (WMMA) | naive WMMA -> shared-mem tiled -> double-buffered | cuBLAS FP16 |
| Softmax | naive 3-kernel -> fused single-pass (warp shuffles) | torch.softmax |

Every kernel is verified (vs cuBLAS / a double-precision CPU reference / PyTorch) before it is timed.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
!mkdir -p src ext python results

## Source files

In [ ]:
%%writefile src/common.cuh
// common.cuh - error checking, timing, init, verification helpers
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <algorithm>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CUDA_CHECK(call)                                                      \
  do {                                                                        \
    cudaError_t err_ = (call);                                                \
    if (err_ != cudaSuccess) {                                                \
      fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_),   \
              __FILE__, __LINE__);                                            \
      exit(1);                                                                \
    }                                                                         \
  } while (0)

#define CUBLAS_CHECK(call)                                                    \
  do {                                                                        \
    cublasStatus_t st_ = (call);                                              \
    if (st_ != CUBLAS_STATUS_SUCCESS) {                                       \
      fprintf(stderr, "cuBLAS error %d at %s:%d\n", (int)st_, __FILE__,       \
              __LINE__);                                                      \
      exit(1);                                                                \
    }                                                                         \
  } while (0)

inline int ceil_div(int a, int b) { return (a + b - 1) / b; }

// Uniform random in [-1, 1], fixed seed for reproducibility.
inline void fill_random(std::vector<float>& v, unsigned seed) {
  srand(seed);
  for (auto& x : v) x = 2.0f * (float)rand() / (float)RAND_MAX - 1.0f;
}

// Max relative error vs reference. Different summation order => small FP32 diffs.
inline float max_rel_err(const std::vector<float>& out,
                         const std::vector<float>& ref) {
  float worst = 0.f;
  for (size_t i = 0; i < out.size(); ++i) {
    float e = fabsf(out[i] - ref[i]) / (1.0f + fabsf(ref[i]));
    worst = std::max(worst, e);
  }
  return worst;
}

// Times `fn` (which launches GPU work) with CUDA events.
// Returns the median of `iters` runs in milliseconds, after `warmup` runs.
template <typename F>
float time_ms(F fn, int warmup = 3, int iters = 10) {
  for (int i = 0; i < warmup; ++i) fn();
  CUDA_CHECK(cudaDeviceSynchronize());
  cudaEvent_t start, stop;
  CUDA_CHECK(cudaEventCreate(&start));
  CUDA_CHECK(cudaEventCreate(&stop));
  std::vector<float> t(iters);
  for (int i = 0; i < iters; ++i) {
    CUDA_CHECK(cudaEventRecord(start));
    fn();
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&t[i], start, stop));
  }
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaEventDestroy(start));
  CUDA_CHECK(cudaEventDestroy(stop));
  std::sort(t.begin(), t.end());
  return t[iters / 2];
}


In [ ]:
%%writefile src/kernels.cuh
// kernels.cuh - SGEMM optimization ladder.
// All matrices are row-major. C = alpha * A(MxK) * B(KxN) + beta * C(MxN).
#pragma once
#include "common.cuh"

// ---------------------------------------------------------------------------
// Stage 0: naive, UNCOALESCED.
// threadIdx.x walks down ROWS, so the 32 threads of a warp read A and write C
// with a stride of K / N floats -> each access is its own memory transaction.
// ---------------------------------------------------------------------------
__global__ void sgemm_naive(int M, int N, int K, float alpha, const float* A,
                            const float* B, float beta, float* C) {
  const int row = blockIdx.x * blockDim.x + threadIdx.x;
  const int col = blockIdx.y * blockDim.y + threadIdx.y;
  if (row < M && col < N) {
    float acc = 0.f;
    for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = alpha * acc + beta * C[row * N + col];
  }
}

// ---------------------------------------------------------------------------
// Stage 1: global memory coalescing.
// Same math; threadIdx.x now walks along COLUMNS, so a warp reads 32
// consecutive floats of B / writes 32 consecutive floats of C (one 128B line),
// and all 32 threads read the SAME A element (a broadcast).
// ---------------------------------------------------------------------------
__global__ void sgemm_coalesced(int M, int N, int K, float alpha,
                                const float* A, const float* B, float beta,
                                float* C) {
  const int col = blockIdx.x * blockDim.x + threadIdx.x;
  const int row = blockIdx.y * blockDim.y + threadIdx.y;
  if (row < M && col < N) {
    float acc = 0.f;
    for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = alpha * acc + beta * C[row * N + col];
  }
}

// ---------------------------------------------------------------------------
// Stage 2: shared-memory tiling.
// A 32x32 block loads a 32x32 tile of A and of B into shared memory, then
// every thread does 32 FMAs out of shared memory. Each global element is now
// loaded once per tile instead of once per FMA -> global traffic / 32.
// Inner loop: As[ty][k] is a broadcast (one ty per warp), Bs[k][tx] hits 32
// consecutive banks -> no bank conflicts.
// Bottleneck left: 2 shared loads per FMA, so it becomes shared-memory bound.
// ---------------------------------------------------------------------------
template <int TILE>
__global__ void sgemm_smem(int M, int N, int K, float alpha, const float* A,
                           const float* B, float beta, float* C) {
  __shared__ float As[TILE][TILE];
  __shared__ float Bs[TILE][TILE];
  const int tx = threadIdx.x, ty = threadIdx.y;
  const int row = blockIdx.y * TILE + ty;
  const int col = blockIdx.x * TILE + tx;

  float acc = 0.f;
  for (int k0 = 0; k0 < K; k0 += TILE) {
    As[ty][tx] = (row < M && k0 + tx < K) ? A[row * K + k0 + tx] : 0.f;
    Bs[ty][tx] = (k0 + ty < K && col < N) ? B[(k0 + ty) * N + col] : 0.f;
    __syncthreads();
#pragma unroll
    for (int k = 0; k < TILE; ++k) acc += As[ty][k] * Bs[k][tx];
    __syncthreads();
  }
  if (row < M && col < N) C[row * N + col] = alpha * acc + beta * C[row * N + col];
}

// ---------------------------------------------------------------------------
// Stage 3: 2D register blocking.
// Block tile BM x BN, K-step BK, each thread computes a TM x TN micro-tile
// held in registers. Per k-step a thread loads TM values of A and TN of B into
// registers and does TM*TN FMAs: 16 shared loads feed 64 FMAs (was 2 per 1).
// Requires M % BM == 0, N % BN == 0, K % BK == 0.
// ---------------------------------------------------------------------------
template <int BM, int BN, int BK, int TM, int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN))
sgemm_regblock(int M, int N, int K, float alpha, const float* A,
               const float* B, float beta, float* C) {
  constexpr int THREADS = (BM * BN) / (TM * TN);
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  const int tid = threadIdx.x;
  const int threadRow = tid / (BN / TN);
  const int threadCol = tid % (BN / TN);

  A += blockIdx.y * BM * K;
  B += blockIdx.x * BN;
  C += blockIdx.y * BM * N + blockIdx.x * BN;

  // Cooperative load mapping (coalesced along the row of each tile).
  const int rowA = tid / BK, colA = tid % BK;
  constexpr int strideA = THREADS / BK;
  const int rowB = tid / BN, colB = tid % BN;
  constexpr int strideB = THREADS / BN;

  float acc[TM * TN] = {0.f};
  float regM[TM], regN[TN];

  for (int k0 = 0; k0 < K; k0 += BK) {
#pragma unroll
    for (int off = 0; off < BM; off += strideA)
      As[(rowA + off) * BK + colA] = A[(rowA + off) * K + colA];
#pragma unroll
    for (int off = 0; off < BK; off += strideB)
      Bs[(rowB + off) * BN + colB] = B[(rowB + off) * N + colB];
    __syncthreads();
    A += BK;
    B += BK * N;

#pragma unroll
    for (int k = 0; k < BK; ++k) {
#pragma unroll
      for (int i = 0; i < TM; ++i) regM[i] = As[(threadRow * TM + i) * BK + k];
#pragma unroll
      for (int j = 0; j < TN; ++j) regN[j] = Bs[k * BN + threadCol * TN + j];
#pragma unroll
      for (int i = 0; i < TM; ++i)
#pragma unroll
        for (int j = 0; j < TN; ++j) acc[i * TN + j] += regM[i] * regN[j];
    }
    __syncthreads();
  }

#pragma unroll
  for (int i = 0; i < TM; ++i)
#pragma unroll
    for (int j = 0; j < TN; ++j) {
      float* c = &C[(threadRow * TM + i) * N + threadCol * TN + j];
      *c = alpha * acc[i * TN + j] + beta * *c;
    }
}

// ---------------------------------------------------------------------------
// Stage 4: vectorized float4 loads/stores + transposed A tile.
// Same tiling as stage 3, but:
//  - A and B are read from global as float4 (4x fewer load instructions,
//    128-bit transactions);
//  - the A tile is stored TRANSPOSED in shared memory (As[k][m]) so each
//    thread's TM values for one k are contiguous and can be read as float4;
//  - C is read/written as float4.
// Requires M % BM == 0, N % BN == 0, K % BK == 0, and N, K multiples of 4.
// ---------------------------------------------------------------------------
template <int BM, int BN, int BK, int TM, int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN))
sgemm_vec4(int M, int N, int K, float alpha, const float* A, const float* B,
           float beta, float* C) {
  constexpr int THREADS = (BM * BN) / (TM * TN);
  static_assert(THREADS * 4 == BM * BK, "one float4 of A per thread per k-step");
  static_assert(THREADS * 4 == BK * BN, "one float4 of B per thread per k-step");
  __shared__ __align__(16) float As[BK * BM];  // transposed: As[k * BM + m]
  __shared__ __align__(16) float Bs[BK * BN];

  const int tid = threadIdx.x;
  const int threadRow = tid / (BN / TN);
  const int threadCol = tid % (BN / TN);

  A += blockIdx.y * BM * K;
  B += blockIdx.x * BN;
  C += blockIdx.y * BM * N + blockIdx.x * BN;

  const int rowA = tid / (BK / 4), colA = tid % (BK / 4);
  const int rowB = tid / (BN / 4), colB = tid % (BN / 4);

  float acc[TM * TN] = {0.f};
  float regM[TM], regN[TN];

  for (int k0 = 0; k0 < K; k0 += BK) {
    const float4 a4 = *reinterpret_cast<const float4*>(&A[rowA * K + colA * 4]);
    As[(colA * 4 + 0) * BM + rowA] = a4.x;
    As[(colA * 4 + 1) * BM + rowA] = a4.y;
    As[(colA * 4 + 2) * BM + rowA] = a4.z;
    As[(colA * 4 + 3) * BM + rowA] = a4.w;
    *reinterpret_cast<float4*>(&Bs[rowB * BN + colB * 4]) =
        *reinterpret_cast<const float4*>(&B[rowB * N + colB * 4]);
    __syncthreads();
    A += BK;
    B += BK * N;

#pragma unroll
    for (int k = 0; k < BK; ++k) {
#pragma unroll
      for (int i = 0; i < TM; i += 4) {
        const float4 t = *reinterpret_cast<const float4*>(&As[k * BM + threadRow * TM + i]);
        regM[i] = t.x; regM[i + 1] = t.y; regM[i + 2] = t.z; regM[i + 3] = t.w;
      }
#pragma unroll
      for (int j = 0; j < TN; j += 4) {
        const float4 t = *reinterpret_cast<const float4*>(&Bs[k * BN + threadCol * TN + j]);
        regN[j] = t.x; regN[j + 1] = t.y; regN[j + 2] = t.z; regN[j + 3] = t.w;
      }
#pragma unroll
      for (int i = 0; i < TM; ++i)
#pragma unroll
        for (int j = 0; j < TN; ++j) acc[i * TN + j] += regM[i] * regN[j];
    }
    __syncthreads();
  }

#pragma unroll
  for (int i = 0; i < TM; ++i)
#pragma unroll
    for (int j = 0; j < TN; j += 4) {
      float4* cp = reinterpret_cast<float4*>(
          &C[(threadRow * TM + i) * N + threadCol * TN + j]);
      float4 c = *cp;
      c.x = alpha * acc[i * TN + j + 0] + beta * c.x;
      c.y = alpha * acc[i * TN + j + 1] + beta * c.y;
      c.z = alpha * acc[i * TN + j + 2] + beta * c.z;
      c.w = alpha * acc[i * TN + j + 3] + beta * c.w;
      *cp = c;
    }
}

// ---------------------------------------------------------------------------
// Launchers. Return false if the size is not supported by the kernel's tiling.
// ---------------------------------------------------------------------------
namespace rb {  // tile config shared by stages 3 and 4: 256 threads, 8x8 per thread
constexpr int BM = 128, BN = 128, BK = 8, TM = 8, TN = 8;
constexpr int THREADS = (BM * BN) / (TM * TN);
}  // namespace rb

inline bool launch_naive(int M, int N, int K, float alpha, const float* A,
                         const float* B, float beta, float* C) {
  dim3 block(32, 32);
  dim3 grid(ceil_div(M, 32), ceil_div(N, 32));
  sgemm_naive<<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  return true;
}

inline bool launch_coalesced(int M, int N, int K, float alpha, const float* A,
                             const float* B, float beta, float* C) {
  dim3 block(32, 32);
  dim3 grid(ceil_div(N, 32), ceil_div(M, 32));
  sgemm_coalesced<<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  return true;
}

inline bool launch_smem(int M, int N, int K, float alpha, const float* A,
                        const float* B, float beta, float* C) {
  dim3 block(32, 32);
  dim3 grid(ceil_div(N, 32), ceil_div(M, 32));
  sgemm_smem<32><<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  return true;
}

inline bool launch_regblock(int M, int N, int K, float alpha, const float* A,
                            const float* B, float beta, float* C) {
  using namespace rb;
  if (M % BM || N % BN || K % BK) return false;
  dim3 grid(N / BN, M / BM);
  sgemm_regblock<BM, BN, BK, TM, TN><<<grid, THREADS>>>(M, N, K, alpha, A, B, beta, C);
  return true;
}

inline bool launch_vec4(int M, int N, int K, float alpha, const float* A,
                        const float* B, float beta, float* C) {
  using namespace rb;
  if (M % BM || N % BN || K % BK) return false;
  dim3 grid(N / BN, M / BM);
  sgemm_vec4<BM, BN, BK, TM, TN><<<grid, THREADS>>>(M, N, K, alpha, A, B, beta, C);
  return true;
}

// cuBLAS is column-major. Row-major C = A*B is column-major C^T = B^T * A^T,
// so pass B first and swap M/N; no transposes needed.
inline void launch_cublas(cublasHandle_t h, int M, int N, int K, float alpha,
                          const float* A, const float* B, float beta,
                          float* C) {
  CUBLAS_CHECK(cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, B, N,
                           A, K, &beta, C, N));
}


In [ ]:
%%writefile src/kernels_wmma.cuh
// kernels_wmma.cuh - FP16 tensor-core GEMM via WMMA (sm_70+; T4 = sm_75).
// Mixed precision: A, B in FP16, accumulate + output in FP32. Row-major.
// C = A(MxK) * B(KxN)
#pragma once
#include <mma.h>
#include <cuda_fp16.h>
#include "common.cuh"

using namespace nvcuda;

// ---------------------------------------------------------------------------
// TC-0: naive WMMA. One warp computes one 16x16 tile of C, loading fragments
// straight from global memory. No data reuse between warps.
// Block = 4 warps arranged 2x2 -> 32x32 tile of C per block.
// Requires M, N, K multiples of 16.
// ---------------------------------------------------------------------------
__global__ void hgemm_wmma_naive(int M, int N, int K, const half* A,
                                 const half* B, float* C) {
  const int warp = threadIdx.x / 32;
  const int tileRow = (blockIdx.y * 2 + warp / 2) * 16;
  const int tileCol = (blockIdx.x * 2 + warp % 2) * 16;
  if (tileRow >= M || tileCol >= N) return;  // warp-uniform exit

  wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a;
  wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b;
  wmma::fragment<wmma::accumulator, 16, 16, 16, float> c;
  wmma::fill_fragment(c, 0.f);

  for (int k = 0; k < K; k += 16) {
    wmma::load_matrix_sync(a, A + tileRow * K + k, K);
    wmma::load_matrix_sync(b, B + k * N + tileCol, N);
    wmma::mma_sync(c, a, b, c);
  }
  wmma::store_matrix_sync(C + tileRow * N + tileCol, c, N, wmma::mem_row_major);
}

// ---------------------------------------------------------------------------
// TC-1: shared-memory tiled WMMA.
// Block tile 128x128, K-step 32, 8 warps (2 x 4). Each warp owns a 64x32
// sub-tile = 4x2 accumulator fragments, so every A fragment it loads from
// shared memory is reused 2x and every B fragment 4x.
// Global->shared copies use 16-byte (8 x half) vector loads.
// Shared rows are padded by 8 halfs to break bank conflicts while keeping
// the 32-byte fragment alignment WMMA needs.
// Requires M % 128 == 0, N % 128 == 0, K % 32 == 0.
// ---------------------------------------------------------------------------
namespace tc1 {
constexpr int BM = 128, BN = 128, BK = 32, PAD = 8;
constexpr int WARPS_M = 2, WARPS_N = 4;          // 8 warps
constexpr int WM = BM / WARPS_M, WN = BN / WARPS_N;  // 64 x 32 per warp
constexpr int FM = WM / 16, FN = WN / 16;          // 4 x 2 fragments
constexpr int THREADS = WARPS_M * WARPS_N * 32;    // 256
}  // namespace tc1

__global__ void __launch_bounds__(tc1::THREADS)
hgemm_wmma_smem(int M, int N, int K, const half* A, const half* B, float* C) {
  using namespace tc1;
  __shared__ __align__(32) half As[BM][BK + PAD];
  __shared__ __align__(32) half Bs[BK][BN + PAD];

  const int tid = threadIdx.x;
  const int warp = tid / 32;
  const int wr = warp / WARPS_N;  // 0..1
  const int wc = warp % WARPS_N;  // 0..3
  const int bm = blockIdx.y * BM;
  const int bn = blockIdx.x * BN;

  wmma::fragment<wmma::accumulator, 16, 16, 16, float> acc[FM][FN];
#pragma unroll
  for (int i = 0; i < FM; ++i)
#pragma unroll
    for (int j = 0; j < FN; ++j) wmma::fill_fragment(acc[i][j], 0.f);

  wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a[FM];
  wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b[FN];

  for (int k0 = 0; k0 < K; k0 += BK) {
    // A tile: 128 x 32 halfs = 512 int4 vectors, 2 per thread (4 per row).
#pragma unroll
    for (int i = 0; i < 2; ++i) {
      const int v = tid + i * THREADS;
      const int row = v / (BK / 8), col = (v % (BK / 8)) * 8;
      *reinterpret_cast<int4*>(&As[row][col]) =
          *reinterpret_cast<const int4*>(&A[(bm + row) * K + k0 + col]);
    }
    // B tile: 32 x 128 halfs = 512 int4 vectors, 2 per thread (16 per row).
#pragma unroll
    for (int i = 0; i < 2; ++i) {
      const int v = tid + i * THREADS;
      const int row = v / (BN / 8), col = (v % (BN / 8)) * 8;
      *reinterpret_cast<int4*>(&Bs[row][col]) =
          *reinterpret_cast<const int4*>(&B[(k0 + row) * N + bn + col]);
    }
    __syncthreads();

#pragma unroll
    for (int kk = 0; kk < BK; kk += 16) {
#pragma unroll
      for (int i = 0; i < FM; ++i)
        wmma::load_matrix_sync(a[i], &As[wr * WM + i * 16][kk], BK + PAD);
#pragma unroll
      for (int j = 0; j < FN; ++j)
        wmma::load_matrix_sync(b[j], &Bs[kk][wc * WN + j * 16], BN + PAD);
#pragma unroll
      for (int i = 0; i < FM; ++i)
#pragma unroll
        for (int j = 0; j < FN; ++j) wmma::mma_sync(acc[i][j], a[i], b[j], acc[i][j]);
    }
    __syncthreads();
  }

#pragma unroll
  for (int i = 0; i < FM; ++i)
#pragma unroll
    for (int j = 0; j < FN; ++j) {
      const int r = bm + wr * WM + i * 16;
      const int c = bn + wc * WN + j * 16;
      wmma::store_matrix_sync(C + r * N + c, acc[i][j], N, wmma::mem_row_major);
    }
}

// ---------------------------------------------------------------------------
// TC-2: double-buffered WMMA (software pipelining).
// Same tiling as TC-1, but two shared-memory buffers. While the tensor cores
// work on tile k from buffer `buf`, each thread already has tile k+1 in flight
// from global memory into registers; it is written to buffer `buf ^ 1` after
// the MMAs, so global-load latency hides behind compute. One __syncthreads per
// k-step instead of two. (T4 / sm_75 has no cp.async, so the prefetch goes
// through registers.)
// ---------------------------------------------------------------------------
__global__ void __launch_bounds__(tc1::THREADS)
hgemm_wmma_dbuf(int M, int N, int K, const half* A, const half* B, float* C) {
  using namespace tc1;
  __shared__ __align__(32) half As[2][BM][BK + PAD];
  __shared__ __align__(32) half Bs[2][BK][BN + PAD];

  const int tid = threadIdx.x;
  const int warp = tid / 32;
  const int wr = warp / WARPS_N;
  const int wc = warp % WARPS_N;
  const int bm = blockIdx.y * BM;
  const int bn = blockIdx.x * BN;

  // Per-thread load slots: 2 int4 of A and 2 int4 of B per k-step.
  int aRow[2], aCol[2], bRow[2], bCol[2];
#pragma unroll
  for (int i = 0; i < 2; ++i) {
    const int v = tid + i * THREADS;
    aRow[i] = v / (BK / 8); aCol[i] = (v % (BK / 8)) * 8;
    bRow[i] = v / (BN / 8); bCol[i] = (v % (BN / 8)) * 8;
  }
  int4 ra[2], rb[2];
  auto load_global = [&](int k0) {
#pragma unroll
    for (int i = 0; i < 2; ++i) {
      ra[i] = *reinterpret_cast<const int4*>(&A[(bm + aRow[i]) * K + k0 + aCol[i]]);
      rb[i] = *reinterpret_cast<const int4*>(&B[(k0 + bRow[i]) * N + bn + bCol[i]]);
    }
  };
  auto store_shared = [&](int buf) {
#pragma unroll
    for (int i = 0; i < 2; ++i) {
      *reinterpret_cast<int4*>(&As[buf][aRow[i]][aCol[i]]) = ra[i];
      *reinterpret_cast<int4*>(&Bs[buf][bRow[i]][bCol[i]]) = rb[i];
    }
  };

  wmma::fragment<wmma::accumulator, 16, 16, 16, float> acc[FM][FN];
#pragma unroll
  for (int i = 0; i < FM; ++i)
#pragma unroll
    for (int j = 0; j < FN; ++j) wmma::fill_fragment(acc[i][j], 0.f);
  wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a[FM];
  wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b[FN];

  // Prologue: tile 0 -> buffer 0.
  load_global(0);
  store_shared(0);
  __syncthreads();

  int buf = 0;
  for (int k0 = 0; k0 < K; k0 += BK) {
    const bool has_next = k0 + BK < K;
    if (has_next) load_global(k0 + BK);  // in flight during the MMAs below

#pragma unroll
    for (int kk = 0; kk < BK; kk += 16) {
#pragma unroll
      for (int i = 0; i < FM; ++i)
        wmma::load_matrix_sync(a[i], &As[buf][wr * WM + i * 16][kk], BK + PAD);
#pragma unroll
      for (int j = 0; j < FN; ++j)
        wmma::load_matrix_sync(b[j], &Bs[buf][kk][wc * WN + j * 16], BN + PAD);
#pragma unroll
      for (int i = 0; i < FM; ++i)
#pragma unroll
        for (int j = 0; j < FN; ++j) wmma::mma_sync(acc[i][j], a[i], b[j], acc[i][j]);
    }

    // buf^1 was last read in the previous k-step, which every warp finished
    // before the previous __syncthreads, so it is safe to overwrite now.
    if (has_next) store_shared(buf ^ 1);
    __syncthreads();
    buf ^= 1;
  }

#pragma unroll
  for (int i = 0; i < FM; ++i)
#pragma unroll
    for (int j = 0; j < FN; ++j) {
      const int r = bm + wr * WM + i * 16;
      const int c = bn + wc * WN + j * 16;
      wmma::store_matrix_sync(C + r * N + c, acc[i][j], N, wmma::mem_row_major);
    }
}

// ---------------------------------------------------------------------------
// Launchers
// ---------------------------------------------------------------------------
inline bool launch_wmma_naive(int M, int N, int K, const half* A,
                              const half* B, float* C) {
  if (M % 16 || N % 16 || K % 16) return false;
  dim3 grid(ceil_div(N, 32), ceil_div(M, 32));
  hgemm_wmma_naive<<<grid, 128>>>(M, N, K, A, B, C);
  return true;
}

inline bool launch_wmma_smem(int M, int N, int K, const half* A,
                             const half* B, float* C) {
  if (M % tc1::BM || N % tc1::BN || K % tc1::BK) return false;
  dim3 grid(N / tc1::BN, M / tc1::BM);
  hgemm_wmma_smem<<<grid, tc1::THREADS>>>(M, N, K, A, B, C);
  return true;
}

inline bool launch_wmma_dbuf(int M, int N, int K, const half* A,
                             const half* B, float* C) {
  if (M % tc1::BM || N % tc1::BN || K % tc1::BK) return false;
  dim3 grid(N / tc1::BN, M / tc1::BM);
  hgemm_wmma_dbuf<<<grid, tc1::THREADS>>>(M, N, K, A, B, C);
  return true;
}

// cuBLAS FP16 inputs, FP32 accumulate + output, tensor cores. Row-major trick
// as in SGEMM: compute C^T = B^T * A^T in column-major.
inline void launch_cublas_fp16(cublasHandle_t h, int M, int N, int K,
                               const half* A, const half* B, float* C) {
  const float alpha = 1.f, beta = 0.f;
  CUBLAS_CHECK(cublasGemmEx(h, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, B,
                            CUDA_R_16F, N, A, CUDA_R_16F, K, &beta, C,
                            CUDA_R_32F, N, CUBLAS_COMPUTE_32F,
                            CUBLAS_GEMM_DEFAULT));
}


In [ ]:
%%writefile src/kernels_softmax.cuh
// kernels_softmax.cuh - row-wise softmax over a (rows x cols) row-major matrix.
// y[r][c] = exp(x[r][c] - max_r) / sum_c' exp(x[r][c'] - max_r)
#pragma once
#include "common.cuh"

constexpr int SM_THREADS = 256;  // one block per row
constexpr int SM_WARPS = SM_THREADS / 32;

__device__ __forceinline__ float warp_max(float v) {
#pragma unroll
  for (int o = 16; o > 0; o >>= 1) v = fmaxf(v, __shfl_xor_sync(0xffffffffu, v, o));
  return v;
}
__device__ __forceinline__ float warp_sum(float v) {
#pragma unroll
  for (int o = 16; o > 0; o >>= 1) v += __shfl_xor_sync(0xffffffffu, v, o);
  return v;
}

// Block-wide reductions: shuffle inside each warp, then warp 0 reduces the
// per-warp partials. `sh` must hold SM_WARPS floats. Every thread gets the result.
__device__ __forceinline__ float block_max(float v, float* sh) {
  const int lane = threadIdx.x & 31, w = threadIdx.x >> 5;
  v = warp_max(v);
  if (lane == 0) sh[w] = v;
  __syncthreads();
  if (w == 0) {
    v = lane < SM_WARPS ? sh[lane] : -INFINITY;
    v = warp_max(v);
    if (lane == 0) sh[0] = v;
  }
  __syncthreads();
  v = sh[0];
  __syncthreads();  // sh is reused by the next reduction
  return v;
}
__device__ __forceinline__ float block_sum(float v, float* sh) {
  const int lane = threadIdx.x & 31, w = threadIdx.x >> 5;
  v = warp_sum(v);
  if (lane == 0) sh[w] = v;
  __syncthreads();
  if (w == 0) {
    v = lane < SM_WARPS ? sh[lane] : 0.f;
    v = warp_sum(v);
    if (lane == 0) sh[0] = v;
  }
  __syncthreads();
  v = sh[0];
  __syncthreads();
  return v;
}

// ---------------------------------------------------------------------------
// Naive: 3 kernels, 3 full passes over x in global memory.
//   1) row max (shared-memory tree reduction)   reads x
//   2) row sum of exp(x - max)                  reads x
//   3) normalize                                reads x, writes y
// Traffic ~ 4 * rows * cols * 4 bytes.
// ---------------------------------------------------------------------------
__global__ void softmax_rowmax(const float* x, float* rowmax, int cols) {
  __shared__ float sh[SM_THREADS];
  const float* xr = x + (size_t)blockIdx.x * cols;
  float m = -INFINITY;
  for (int c = threadIdx.x; c < cols; c += SM_THREADS) m = fmaxf(m, xr[c]);
  sh[threadIdx.x] = m;
  __syncthreads();
  for (int s = SM_THREADS / 2; s > 0; s >>= 1) {  // classic tree reduction
    if (threadIdx.x < s) sh[threadIdx.x] = fmaxf(sh[threadIdx.x], sh[threadIdx.x + s]);
    __syncthreads();
  }
  if (threadIdx.x == 0) rowmax[blockIdx.x] = sh[0];
}

__global__ void softmax_rowsum(const float* x, const float* rowmax,
                               float* rowsum, int cols) {
  __shared__ float sh[SM_THREADS];
  const float* xr = x + (size_t)blockIdx.x * cols;
  const float m = rowmax[blockIdx.x];
  float s = 0.f;
  for (int c = threadIdx.x; c < cols; c += SM_THREADS) s += expf(xr[c] - m);
  sh[threadIdx.x] = s;
  __syncthreads();
  for (int st = SM_THREADS / 2; st > 0; st >>= 1) {
    if (threadIdx.x < st) sh[threadIdx.x] += sh[threadIdx.x + st];
    __syncthreads();
  }
  if (threadIdx.x == 0) rowsum[blockIdx.x] = sh[0];
}

__global__ void softmax_normalize(const float* x, const float* rowmax,
                                  const float* rowsum, float* y, int cols) {
  const size_t base = (size_t)blockIdx.x * cols;
  const float m = rowmax[blockIdx.x];
  const float inv = 1.f / rowsum[blockIdx.x];
  for (int c = threadIdx.x; c < cols; c += SM_THREADS)
    y[base + c] = expf(x[base + c] - m) * inv;
}

// ---------------------------------------------------------------------------
// Fused: 1 kernel, 1 read of x + 1 write of y.
// The whole row is held in registers (VPT values per thread, coalesced
// stride-SM_THREADS layout), max and sum use warp-shuffle block reductions,
// and the exp is computed once and reused for the output.
// Traffic ~ 2 * rows * cols * 4 bytes. Supports cols <= VPT * SM_THREADS.
// ---------------------------------------------------------------------------
template <int VPT>
__global__ void __launch_bounds__(SM_THREADS)
softmax_fused(const float* x, float* y, int cols) {
  __shared__ float sh[SM_WARPS];
  const size_t base = (size_t)blockIdx.x * cols;
  float v[VPT];
  float m = -INFINITY;
#pragma unroll
  for (int i = 0; i < VPT; ++i) {
    const int c = threadIdx.x + i * SM_THREADS;
    v[i] = c < cols ? x[base + c] : -INFINITY;
    m = fmaxf(m, v[i]);
  }
  m = block_max(m, sh);

  float s = 0.f;
#pragma unroll
  for (int i = 0; i < VPT; ++i) {
    const int c = threadIdx.x + i * SM_THREADS;
    v[i] = c < cols ? expf(v[i] - m) : 0.f;
    s += v[i];
  }
  s = block_sum(s, sh);

  const float inv = 1.f / s;
#pragma unroll
  for (int i = 0; i < VPT; ++i) {
    const int c = threadIdx.x + i * SM_THREADS;
    if (c < cols) y[base + c] = v[i] * inv;
  }
}

// ---------------------------------------------------------------------------
// Launchers
// ---------------------------------------------------------------------------
// scratch must hold 2 * rows floats (row max + row sum).
inline void launch_softmax_naive(const float* x, float* y, float* scratch,
                                 int rows, int cols) {
  float* rowmax = scratch;
  float* rowsum = scratch + rows;
  softmax_rowmax<<<rows, SM_THREADS>>>(x, rowmax, cols);
  softmax_rowsum<<<rows, SM_THREADS>>>(x, rowmax, rowsum, cols);
  softmax_normalize<<<rows, SM_THREADS>>>(x, rowmax, rowsum, y, cols);
}

// Returns false if cols is too wide to keep a row in registers.
inline bool launch_softmax_fused(const float* x, float* y, int rows, int cols) {
  const int vpt = ceil_div(cols, SM_THREADS);
  if (vpt <= 1)       softmax_fused<1><<<rows, SM_THREADS>>>(x, y, cols);
  else if (vpt <= 2)  softmax_fused<2><<<rows, SM_THREADS>>>(x, y, cols);
  else if (vpt <= 4)  softmax_fused<4><<<rows, SM_THREADS>>>(x, y, cols);
  else if (vpt <= 8)  softmax_fused<8><<<rows, SM_THREADS>>>(x, y, cols);
  else if (vpt <= 16) softmax_fused<16><<<rows, SM_THREADS>>>(x, y, cols);
  else return false;
  return true;
}


In [ ]:
%%writefile src/sgemm.cu
// sgemm.cu - benchmark + correctness harness for the SGEMM ladder.
// Usage: ./sgemm [size ...]        (default: 1024 2048 4096)
// Prints a CSV row per (kernel, size) and appends it to results/sgemm.csv.
#include <string>
#include <functional>
#include "kernels.cuh"

struct Kernel {
  std::string name;
  std::function<bool(int, int, int, float, const float*, const float*, float,
                     float*)> run;
};

int main(int argc, char** argv) {
  std::vector<int> sizes = {1024, 2048, 4096};
  if (argc > 1) {
    sizes.clear();
    for (int i = 1; i < argc; ++i) sizes.push_back(atoi(argv[i]));
  }

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("GPU: %s | SMs: %d | sm_%d%d\n", prop.name, prop.multiProcessorCount,
         prop.major, prop.minor);

  cublasHandle_t handle;
  CUBLAS_CHECK(cublasCreate(&handle));
  // Keep cuBLAS on plain FP32 (no TF32 / tensor-core shortcuts) for a fair baseline.
  CUBLAS_CHECK(cublasSetMathMode(handle, CUBLAS_DEFAULT_MATH));

  std::vector<Kernel> kernels = {
      {"0_naive", launch_naive},
      {"1_coalesced", launch_coalesced},
      {"2_smem_tiling", launch_smem},
      {"3_reg_blocking", launch_regblock},
      {"4_vec4", launch_vec4},
  };

  FILE* csv = fopen("results/sgemm.csv", "a");
  printf("%-15s %6s %10s %10s %9s %10s\n", "kernel", "size", "ms", "GFLOPS",
         "%cuBLAS", "max_err");

  for (int n : sizes) {
    const int M = n, N = n, K = n;
    const float alpha = 1.f, beta = 0.f;
    const double flops = 2.0 * M * N * K;
    const size_t bytesA = sizeof(float) * M * K, bytesB = sizeof(float) * K * N,
                 bytesC = sizeof(float) * M * N;

    std::vector<float> hA(M * K), hB(K * N), hRef(M * N), hOut(M * N);
    fill_random(hA, 1);
    fill_random(hB, 2);

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytesA));
    CUDA_CHECK(cudaMalloc(&dB, bytesB));
    CUDA_CHECK(cudaMalloc(&dC, bytesC));
    CUDA_CHECK(cudaMemcpy(dA, hA.data(), bytesA, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB.data(), bytesB, cudaMemcpyHostToDevice));

    // Reference + baseline: cuBLAS.
    CUDA_CHECK(cudaMemset(dC, 0, bytesC));
    launch_cublas(handle, M, N, K, alpha, dA, dB, beta, dC);
    CUDA_CHECK(cudaMemcpy(hRef.data(), dC, bytesC, cudaMemcpyDeviceToHost));
    float ms_cublas = time_ms(
        [&] { launch_cublas(handle, M, N, K, alpha, dA, dB, beta, dC); });
    double gf_cublas = flops / (ms_cublas * 1e6);
    printf("%-15s %6d %10.3f %10.1f %8.1f%% %10s\n", "cublas", n, ms_cublas,
           gf_cublas, 100.0, "-");
    if (csv) fprintf(csv, "%s,cublas,%d,%.4f,%.1f,100.0,0\n", prop.name, n,
                     ms_cublas, gf_cublas);

    for (auto& kern : kernels) {
      // Naive at 4096 takes seconds per run; fewer iters keeps it bearable.
      int iters = (kern.name == "0_naive" && n >= 4096) ? 3 : 10;
      CUDA_CHECK(cudaMemset(dC, 0, bytesC));
      if (!kern.run(M, N, K, alpha, dA, dB, beta, dC)) {
        printf("%-15s %6d  skipped (size not a multiple of the tile)\n",
               kern.name.c_str(), n);
        continue;
      }
      CUDA_CHECK(cudaGetLastError());
      CUDA_CHECK(cudaMemcpy(hOut.data(), dC, bytesC, cudaMemcpyDeviceToHost));
      float err = max_rel_err(hOut, hRef);

      float ms = time_ms(
          [&] { kern.run(M, N, K, alpha, dA, dB, beta, dC); }, 1, iters);
      double gf = flops / (ms * 1e6);
      printf("%-15s %6d %10.3f %10.1f %8.1f%% %10.2e%s\n", kern.name.c_str(),
             n, ms, gf, 100.0 * gf / gf_cublas, err,
             err > 1e-3f ? "  <-- WRONG" : "");
      if (csv) fprintf(csv, "%s,%s,%d,%.4f,%.1f,%.1f,%.2e\n", prop.name,
                       kern.name.c_str(), n, ms, gf, 100.0 * gf / gf_cublas,
                       err);
    }

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
  }

  if (csv) fclose(csv);
  CUBLAS_CHECK(cublasDestroy(handle));
  return 0;
}


In [ ]:
%%writefile src/hgemm.cu
// hgemm.cu - benchmark + correctness harness for the WMMA tensor-core track.
// Usage: ./hgemm [size ...]        (default: 1024 2048 4096)
// FP16 inputs, FP32 accumulate/output. Reference = cuBLAS GemmEx (tensor cores).
#include <string>
#include <functional>
#include "kernels_wmma.cuh"

struct TcKernel {
  std::string name;
  std::function<bool(int, int, int, const half*, const half*, float*)> run;
};

int main(int argc, char** argv) {
  std::vector<int> sizes = {1024, 2048, 4096};
  if (argc > 1) {
    sizes.clear();
    for (int i = 1; i < argc; ++i) sizes.push_back(atoi(argv[i]));
  }

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("GPU: %s | SMs: %d | sm_%d%d\n", prop.name, prop.multiProcessorCount,
         prop.major, prop.minor);
  if (prop.major < 7) {
    fprintf(stderr, "WMMA needs sm_70+. On Kaggle pick 'GPU T4 x2', not P100.\n");
    return 1;
  }

  cublasHandle_t handle;
  CUBLAS_CHECK(cublasCreate(&handle));

  std::vector<TcKernel> kernels = {
      {"tc0_wmma_naive", launch_wmma_naive},
      {"tc1_wmma_smem", launch_wmma_smem},
      {"tc2_wmma_dbuf", launch_wmma_dbuf},
  };

  FILE* csv = fopen("results/hgemm.csv", "a");
  printf("%-16s %6s %10s %10s %9s %10s\n", "kernel", "size", "ms", "TFLOPS",
         "%cuBLAS", "max_err");

  for (int n : sizes) {
    const int M = n, N = n, K = n;
    const double flops = 2.0 * M * N * K;

    std::vector<float> fA(M * K), fB(K * N);
    fill_random(fA, 1);
    fill_random(fB, 2);
    std::vector<half> hA(M * K), hB(K * N);
    for (size_t i = 0; i < fA.size(); ++i) hA[i] = __float2half(fA[i]);
    for (size_t i = 0; i < fB.size(); ++i) hB[i] = __float2half(fB[i]);
    std::vector<float> hRef(M * N), hOut(M * N);

    half *dA, *dB;
    float* dC;
    CUDA_CHECK(cudaMalloc(&dA, sizeof(half) * M * K));
    CUDA_CHECK(cudaMalloc(&dB, sizeof(half) * K * N));
    CUDA_CHECK(cudaMalloc(&dC, sizeof(float) * M * N));
    CUDA_CHECK(cudaMemcpy(dA, hA.data(), sizeof(half) * M * K, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB.data(), sizeof(half) * K * N, cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemset(dC, 0, sizeof(float) * M * N));
    launch_cublas_fp16(handle, M, N, K, dA, dB, dC);
    CUDA_CHECK(cudaMemcpy(hRef.data(), dC, sizeof(float) * M * N, cudaMemcpyDeviceToHost));
    float ms_ref = time_ms([&] { launch_cublas_fp16(handle, M, N, K, dA, dB, dC); });
    double tf_ref = flops / (ms_ref * 1e9);
    printf("%-16s %6d %10.3f %10.2f %8.1f%% %10s\n", "cublas_fp16", n, ms_ref,
           tf_ref, 100.0, "-");
    if (csv) fprintf(csv, "%s,cublas_fp16,%d,%.4f,%.3f,100.0,0\n", prop.name, n,
                     ms_ref, tf_ref);

    for (auto& kern : kernels) {
      CUDA_CHECK(cudaMemset(dC, 0, sizeof(float) * M * N));
      if (!kern.run(M, N, K, dA, dB, dC)) {
        printf("%-16s %6d  skipped (size not a multiple of the tile)\n",
               kern.name.c_str(), n);
        continue;
      }
      CUDA_CHECK(cudaGetLastError());
      CUDA_CHECK(cudaMemcpy(hOut.data(), dC, sizeof(float) * M * N, cudaMemcpyDeviceToHost));
      float err = max_rel_err(hOut, hRef);

      float ms = time_ms([&] { kern.run(M, N, K, dA, dB, dC); });
      double tf = flops / (ms * 1e9);
      printf("%-16s %6d %10.3f %10.2f %8.1f%% %10.2e%s\n", kern.name.c_str(),
             n, ms, tf, 100.0 * tf / tf_ref, err,
             err > 1e-2f ? "  <-- WRONG" : "");
      if (csv) fprintf(csv, "%s,%s,%d,%.4f,%.3f,%.1f,%.2e\n", prop.name,
                       kern.name.c_str(), n, ms, tf, 100.0 * tf / tf_ref, err);
    }

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
  }

  if (csv) fclose(csv);
  CUBLAS_CHECK(cublasDestroy(handle));
  return 0;
}


In [ ]:
%%writefile src/softmax.cu
// softmax.cu - correctness + benchmark harness: naive 3-kernel vs fused softmax.
// Usage: ./softmax                (default shapes, ~8M elements each)
// Appends to results/softmax.csv. Reports effective bandwidth (GB/s) against the
// minimum traffic of the op (read x once + write y once).
#include <string>
#include "kernels_softmax.cuh"

int main() {
  // (rows, cols): attention-like shapes, rows = batch * heads * seq_len.
  const int shapes[][2] = {{16384, 512}, {8192, 1024}, {4096, 2048}, {2048, 4096}};

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("GPU: %s\n", prop.name);

  FILE* csv = fopen("results/softmax.csv", "a");
  printf("%-14s %6s %6s %9s %9s %8s %10s\n", "kernel", "rows", "cols", "ms",
         "GB/s", "speedup", "max_err");

  for (auto& sh : shapes) {
    const int rows = sh[0], cols = sh[1];
    const size_t n = (size_t)rows * cols;
    std::vector<float> hx(n), hy(n), href(n);
    fill_random(hx, 7);
    for (auto& v : hx) v *= 8.f;  // logits in [-8, 8]

    // CPU reference in double precision.
    for (int r = 0; r < rows; ++r) {
      const float* xr = &hx[(size_t)r * cols];
      double m = -1e300, s = 0;
      for (int c = 0; c < cols; ++c) m = std::max(m, (double)xr[c]);
      for (int c = 0; c < cols; ++c) s += exp((double)xr[c] - m);
      for (int c = 0; c < cols; ++c)
        href[(size_t)r * cols + c] = (float)(exp((double)xr[c] - m) / s);
    }

    float *dx, *dy, *dscratch;
    CUDA_CHECK(cudaMalloc(&dx, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dy, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dscratch, 2 * rows * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(dx, hx.data(), n * sizeof(float), cudaMemcpyHostToDevice));
    const double min_bytes = 2.0 * n * sizeof(float);

    auto check = [&]() {
      CUDA_CHECK(cudaGetLastError());
      CUDA_CHECK(cudaMemcpy(hy.data(), dy, n * sizeof(float), cudaMemcpyDeviceToHost));
      float e = 0.f;  // absolute error: outputs are probabilities in [0, 1]
      for (size_t i = 0; i < n; ++i) e = std::max(e, fabsf(hy[i] - href[i]));
      return e;
    };

    // Naive
    CUDA_CHECK(cudaMemset(dy, 0, n * sizeof(float)));
    launch_softmax_naive(dx, dy, dscratch, rows, cols);
    const float err_naive = check();
    const float ms_naive = time_ms([&] { launch_softmax_naive(dx, dy, dscratch, rows, cols); });
    printf("%-14s %6d %6d %9.4f %9.1f %8s %10.2e%s\n", "naive_3kernel", rows, cols,
           ms_naive, min_bytes / (ms_naive * 1e6), "1.00x", err_naive,
           err_naive > 1e-5f ? "  <-- WRONG" : "");
    if (csv) fprintf(csv, "%s,naive_3kernel,%d,%d,%.5f,%.1f,1.00,%.2e\n", prop.name,
                     rows, cols, ms_naive, min_bytes / (ms_naive * 1e6), err_naive);

    // Fused
    CUDA_CHECK(cudaMemset(dy, 0, n * sizeof(float)));
    if (!launch_softmax_fused(dx, dy, rows, cols)) {
      printf("%-14s %6d %6d  skipped (row too wide for registers)\n", "fused", rows, cols);
    } else {
      const float err_fused = check();
      const float ms_fused = time_ms([&] { launch_softmax_fused(dx, dy, rows, cols); });
      printf("%-14s %6d %6d %9.4f %9.1f %7.2fx %10.2e%s\n", "fused", rows, cols,
             ms_fused, min_bytes / (ms_fused * 1e6), ms_naive / ms_fused, err_fused,
             err_fused > 1e-5f ? "  <-- WRONG" : "");
      if (csv) fprintf(csv, "%s,fused,%d,%d,%.5f,%.1f,%.2f,%.2e\n", prop.name, rows,
                       cols, ms_fused, min_bytes / (ms_fused * 1e6),
                       ms_naive / ms_fused, err_fused);
    }

    CUDA_CHECK(cudaFree(dx));
    CUDA_CHECK(cudaFree(dy));
    CUDA_CHECK(cudaFree(dscratch));
  }
  if (csv) fclose(csv);
  return 0;
}


In [ ]:
%%writefile ext/ops_cuda.cu
// ops_cuda.cu - raw-pointer entry points for the PyTorch extension.
// Kept free of torch headers so nvcc only compiles our kernels; ops.cpp does
// the tensor handling. Each function returns nullptr on success, or an error.
#include "../src/kernels.cuh"
#include "../src/kernels_wmma.cuh"
#include "../src/kernels_softmax.cuh"

static const char* last_cuda_error() {
  cudaError_t e = cudaGetLastError();
  return e == cudaSuccess ? nullptr : cudaGetErrorString(e);
}

// variant: 0 naive, 1 coalesced, 2 smem tiling, 3 register blocking, 4 vec4
const char* dlk_sgemm(int variant, int M, int N, int K, const float* A,
                      const float* B, float* C) {
  const float alpha = 1.f, beta = 0.f;
  bool ok = false;
  switch (variant) {
    case 0: ok = launch_naive(M, N, K, alpha, A, B, beta, C); break;
    case 1: ok = launch_coalesced(M, N, K, alpha, A, B, beta, C); break;
    case 2: ok = launch_smem(M, N, K, alpha, A, B, beta, C); break;
    case 3: ok = launch_regblock(M, N, K, alpha, A, B, beta, C); break;
    case 4: ok = launch_vec4(M, N, K, alpha, A, B, beta, C); break;
    default: return "unknown sgemm variant (expected 0-4)";
  }
  if (!ok) return "matrix size not supported by this variant (needs multiples of 128)";
  return last_cuda_error();
}

// variant: 0 naive WMMA, 1 smem-tiled WMMA, 2 double-buffered WMMA
const char* dlk_hgemm(int variant, int M, int N, int K, const void* A,
                      const void* B, float* C) {
  const half* a = static_cast<const half*>(A);
  const half* b = static_cast<const half*>(B);
  bool ok = false;
  switch (variant) {
    case 0: ok = launch_wmma_naive(M, N, K, a, b, C); break;
    case 1: ok = launch_wmma_smem(M, N, K, a, b, C); break;
    case 2: ok = launch_wmma_dbuf(M, N, K, a, b, C); break;
    default: return "unknown hgemm variant (expected 0-2)";
  }
  if (!ok) return "matrix size not supported by this variant (needs multiples of 16 / 128)";
  return last_cuda_error();
}

// fused: 0 naive 3-kernel (needs scratch of 2*rows floats), 1 fused
const char* dlk_softmax(int fused, const float* x, float* y, float* scratch,
                        int rows, int cols) {
  if (fused) {
    if (!launch_softmax_fused(x, y, rows, cols)) return "cols too large for fused kernel (max 4096)";
  } else {
    launch_softmax_naive(x, y, scratch, rows, cols);
  }
  return last_cuda_error();
}


In [ ]:
%%writefile ext/ops.cpp
// ops.cpp - PyTorch bindings: dlk.sgemm, dlk.hgemm, dlk.softmax
#include <torch/extension.h>

const char* dlk_sgemm(int variant, int M, int N, int K, const float* A,
                      const float* B, float* C);
const char* dlk_hgemm(int variant, int M, int N, int K, const void* A,
                      const void* B, float* C);
const char* dlk_softmax(int fused, const float* x, float* y, float* scratch,
                        int rows, int cols);

static void check_matrix(const torch::Tensor& t, torch::ScalarType dt, const char* name) {
  TORCH_CHECK(t.is_cuda(), name, " must be a CUDA tensor");
  TORCH_CHECK(t.dim() == 2, name, " must be 2-D");
  TORCH_CHECK(t.scalar_type() == dt, name, " has the wrong dtype");
}

// C = A @ B, FP32. variant 0-4 selects the optimization stage.
torch::Tensor sgemm(torch::Tensor A, torch::Tensor B, int64_t variant) {
  check_matrix(A, torch::kFloat32, "A");
  check_matrix(B, torch::kFloat32, "B");
  TORCH_CHECK(A.size(1) == B.size(0), "inner dimensions differ");
  A = A.contiguous();
  B = B.contiguous();
  const int M = A.size(0), K = A.size(1), N = B.size(1);
  auto C = torch::zeros({M, N}, A.options());  // kernels compute alpha*AB + beta*C
  const char* err = dlk_sgemm((int)variant, M, N, K, A.data_ptr<float>(),
                              B.data_ptr<float>(), C.data_ptr<float>());
  TORCH_CHECK(err == nullptr, "sgemm: ", err ? err : "");
  return C;
}

// C = A @ B with FP16 inputs on tensor cores, FP32 accumulate and output.
torch::Tensor hgemm(torch::Tensor A, torch::Tensor B, int64_t variant) {
  check_matrix(A, torch::kFloat16, "A");
  check_matrix(B, torch::kFloat16, "B");
  TORCH_CHECK(A.size(1) == B.size(0), "inner dimensions differ");
  A = A.contiguous();
  B = B.contiguous();
  const int M = A.size(0), K = A.size(1), N = B.size(1);
  auto C = torch::empty({M, N}, A.options().dtype(torch::kFloat32));
  const char* err = dlk_hgemm((int)variant, M, N, K, A.data_ptr(), B.data_ptr(),
                              C.data_ptr<float>());
  TORCH_CHECK(err == nullptr, "hgemm: ", err ? err : "");
  return C;
}

// Row-wise softmax of a 2-D FP32 tensor. fused=true -> single-pass kernel.
torch::Tensor softmax(torch::Tensor x, bool fused) {
  check_matrix(x, torch::kFloat32, "x");
  x = x.contiguous();
  const int rows = x.size(0), cols = x.size(1);
  auto y = torch::empty_like(x);
  auto scratch = torch::empty({2 * rows}, x.options());
  const char* err = dlk_softmax(fused ? 1 : 0, x.data_ptr<float>(), y.data_ptr<float>(),
                                scratch.data_ptr<float>(), rows, cols);
  TORCH_CHECK(err == nullptr, "softmax: ", err ? err : "");
  return y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("sgemm", &sgemm, "FP32 GEMM (variant 0-4)", py::arg("A"), py::arg("B"), py::arg("variant") = 4);
  m.def("hgemm", &hgemm, "FP16 tensor-core GEMM (variant 0-2)", py::arg("A"), py::arg("B"), py::arg("variant") = 2);
  m.def("softmax", &softmax, "row-wise softmax", py::arg("x"), py::arg("fused") = true);
}


In [ ]:
%%writefile python/dlk.py
"""JIT-builds and loads the PyTorch extension (first import takes ~1-2 min)."""
import os
from torch.utils.cpp_extension import load

_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

ops = load(
    name="dlk",
    sources=[os.path.join(_ROOT, "ext", "ops.cpp"), os.path.join(_ROOT, "ext", "ops_cuda.cu")],
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3", "-std=c++17", "-gencode=arch=compute_75,code=sm_75", "-lineinfo"],
    extra_ldflags=["-lcublas"],
    verbose=False,
)


In [ ]:
%%writefile python/test_ops.py
"""Correctness tests of every kernel against PyTorch. Run: python python/test_ops.py"""
import sys
import torch
from dlk import ops

torch.backends.cuda.matmul.allow_tf32 = False  # exact FP32 reference
torch.manual_seed(0)
dev = "cuda"
failures = 0


def report(name, err, tol):
    global failures
    ok = err <= tol
    failures += not ok
    print(f"{'PASS' if ok else 'FAIL'}  {name:<40} max_err={err:.2e} (tol {tol:.0e})")


def rel_err(out, ref):
    return ((out - ref).abs() / (1 + ref.abs())).max().item()


# FP32 SGEMM: square and rectangular shapes (stages 3-4 need multiples of 128).
for (M, N, K) in [(128, 128, 128), (1024, 1024, 1024), (256, 512, 384), (2048, 1024, 512)]:
    A = torch.randn(M, K, device=dev)
    B = torch.randn(K, N, device=dev)
    ref = A @ B
    for v, name in enumerate(["naive", "coalesced", "smem", "regblock", "vec4"]):
        report(f"sgemm[{name}] {M}x{N}x{K}", rel_err(ops.sgemm(A, B, v), ref), 1e-3)

# Odd sizes for the stages that support them.
A = torch.randn(100, 70, device=dev)
B = torch.randn(70, 33, device=dev)
for v, name in enumerate(["naive", "coalesced", "smem"]):
    report(f"sgemm[{name}] 100x33x70 (odd)", rel_err(ops.sgemm(A, B, v), A @ B), 1e-3)

# FP16 tensor-core GEMM. Reference: the same FP16 values multiplied in FP32.
for (M, N, K) in [(128, 128, 128), (1024, 1024, 1024), (256, 512, 384)]:
    A = torch.randn(M, K, device=dev).half()
    B = torch.randn(K, N, device=dev).half()
    ref = A.float() @ B.float()
    for v, name in enumerate(["wmma_naive", "wmma_smem", "wmma_dbuf"]):
        report(f"hgemm[{name}] {M}x{N}x{K}", rel_err(ops.hgemm(A, B, v), ref), 1e-2)

# Softmax vs torch.softmax, including widths that are not multiples of 256.
for (rows, cols) in [(4096, 1024), (37, 1000), (512, 4096), (8, 3), (1024, 2050)]:
    x = torch.randn(rows, cols, device=dev) * 8
    ref = torch.softmax(x, dim=1)
    for fused in (False, True):
        out = ops.softmax(x, fused)
        report(f"softmax[{'fused' if fused else 'naive'}] {rows}x{cols}", (out - ref).abs().max().item(), 1e-5)

print(f"\n{failures} failure(s)")
sys.exit(1 if failures else 0)


In [ ]:
%%writefile python/bench_torch.py
"""Benchmark our kernels against PyTorch through the extension.
Run: python python/bench_torch.py   -> prints a table, writes results/torch_compare.csv"""
import csv
import os
import torch
from dlk import ops

torch.backends.cuda.matmul.allow_tf32 = False
dev = "cuda"


def bench(fn, warmup=3, iters=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record()
        fn()
        e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    times.sort()
    return times[len(times) // 2]  # median ms


rows = []
n = 4096
flops = 2 * n**3

A = torch.randn(n, n, device=dev)
B = torch.randn(n, n, device=dev)
t_ref = bench(lambda: A @ B)
t_ours = bench(lambda: ops.sgemm(A, B, 4))
rows.append(["sgemm fp32 4096", "torch.matmul", t_ref, flops / t_ref / 1e9, "TFLOPS"])
rows.append(["sgemm fp32 4096", "ours (vec4)", t_ours, flops / t_ours / 1e9, "TFLOPS"])

Ah, Bh = A.half(), B.half()
t_ref = bench(lambda: Ah @ Bh)
for v, name in [(1, "ours (wmma_smem)"), (2, "ours (wmma_dbuf)")]:
    t = bench(lambda: ops.hgemm(Ah, Bh, v))
    rows.append(["hgemm fp16 4096", name, t, flops / t / 1e9, "TFLOPS"])
rows.append(["hgemm fp16 4096", "torch.matmul (fp16 out)", t_ref, flops / t_ref / 1e9, "TFLOPS"])

for (r, c) in [(16384, 512), (8192, 1024), (4096, 2048), (2048, 4096)]:
    x = torch.randn(r, c, device=dev)
    gb = 2 * x.numel() * 4 / 1e9  # read once + write once
    for name, fn in [("torch.softmax", lambda: torch.softmax(x, dim=1)),
                     ("ours (naive 3-kernel)", lambda: ops.softmax(x, False)),
                     ("ours (fused)", lambda: ops.softmax(x, True))]:
        t = bench(fn)
        rows.append([f"softmax {r}x{c}", name, t, gb / (t / 1e3), "GB/s"])

print(f"{'op':<20} {'impl':<26} {'ms':>9} {'perf':>9}")
for op, impl, t, perf, unit in rows:
    print(f"{op:<20} {impl:<26} {t:>9.3f} {perf:>8.2f} {unit}")

os.makedirs("results", exist_ok=True)
with open("results/torch_compare.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["op", "impl", "ms", "perf", "unit"])
    w.writerows(rows)


In [ ]:
%%writefile python/roofline.py
"""Analytical performance model + roofline for the GEMM kernels.

For each kernel we derive its global-memory traffic from the tiling (assuming
no cache reuse beyond what the kernel stages itself), giving an arithmetic
intensity AI = FLOPs / bytes. The roofline then predicts a ceiling:
    predicted = min(peak_compute, AI * peak_bandwidth)
and we compare it with the measured number from results/*.csv.

Run: python python/roofline.py  -> table on stdout, results/roofline.png
"""
import os
import pandas as pd
import matplotlib.pyplot as plt

# T4 datasheet peaks. Sustained is lower: the card is power-capped at 70 W.
PEAK_FP32 = 8100.0      # GFLOPS
PEAK_FP16_TC = 65000.0  # GFLOPS, tensor cores, FP32 accumulate
PEAK_BW = 320.0         # GB/s

# Arithmetic intensity (FLOP / byte of global loads) derived from each tiling.
#   2-flop FMA; a C tile of BM x BN reloads (BM + BN) * K elements of A and B.
#   AI = 2 / (bytes_per_elem * (1/BM + 1/BN))
def tile_ai(bm, bn, bytes_per_elem):
    return 2.0 / (bytes_per_elem * (1.0 / bm + 1.0 / bn))

MODEL = {
    # kernel: (track, AI, note)
    "0_naive":        ("FP32", 2.0 / 8.0, "2 global loads per FMA, uncoalesced"),
    "1_coalesced":    ("FP32", 2.0 / 8.0, "2 loads per FMA, but coalesced + L1 broadcast"),
    "2_smem_tiling":  ("FP32", tile_ai(32, 32, 4), "32x32 tile, then shared-memory bound"),
    "3_reg_blocking": ("FP32", tile_ai(128, 128, 4), "128x128 tile, 8x8 per thread"),
    "4_vec4":         ("FP32", tile_ai(128, 128, 4), "same tile, 128-bit loads"),
    "tc0_wmma_naive": ("FP16", tile_ai(16, 16, 2), "one 16x16 tile per warp, no reuse"),
    "tc1_wmma_smem":  ("FP16", tile_ai(128, 128, 2), "128x128 block tile in shared memory"),
    "tc2_wmma_dbuf":  ("FP16", tile_ai(128, 128, 2), "same tile, global loads overlapped"),
}

cols = ["gpu", "kernel", "size", "ms", "perf", "pct_of_cublas", "max_err"]
frames = []
if os.path.exists("results/sgemm.csv"):
    frames.append(pd.read_csv("results/sgemm.csv", names=cols))
if os.path.exists("results/hgemm.csv"):
    h = pd.read_csv("results/hgemm.csv", names=cols)
    h["perf"] = h["perf"] * 1000.0  # TFLOPS -> GFLOPS
    frames.append(h)
df = pd.concat(frames).groupby(["kernel", "size"]).tail(1)
size = df["size"].max()
df = df[df["size"] == size]

rows = []
for k, (track, ai, note) in MODEL.items():
    m = df[df.kernel == k]
    if m.empty:
        continue
    peak = PEAK_FP32 if track == "FP32" else PEAK_FP16_TC
    pred = min(peak, ai * PEAK_BW)
    bound = "memory" if ai * PEAK_BW < peak else "compute"
    meas = float(m.perf.iloc[0])
    rows.append(dict(kernel=k, track=track, AI=round(ai, 2), bound=bound,
                     predicted_gflops=round(pred), measured_gflops=round(meas),
                     pct_of_model=round(100 * meas / pred, 1), note=note))
table = pd.DataFrame(rows)
print(f"Performance model vs measured, {size}^3 GEMM on T4")
print(f"Ridge points: FP32 {PEAK_FP32 / PEAK_BW:.0f} FLOP/B, FP16 tensor {PEAK_FP16_TC / PEAK_BW:.0f} FLOP/B")
print(table.to_string(index=False))
print("pct_of_model > 100 means L1/L2 caches supply reuse the kernel does not stage itself;"
      " < 100 means a bottleneck other than global traffic (shared-memory bandwidth, latency,"
      " occupancy). Confirm which with Nsight Compute.")
table.to_csv("results/perf_model.csv", index=False)

# ---- roofline plot ----
COLORS = {"FP32": "#2a78d6", "FP16": "#eb6834"}  # categorical slots 1, 2
MARKERS = {"FP32": "o", "FP16": "s"}
INK, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e4e3dc"

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
xs = [0.1, 1000.0]
for peak, label in [(PEAK_FP32, "FP32 peak 8.1 TFLOPS"), (PEAK_FP16_TC, "FP16 tensor peak 65 TFLOPS")]:
    ridge = peak / PEAK_BW
    ax.plot([xs[0], ridge, xs[1]], [xs[0] * PEAK_BW, peak, peak], color=MUTED, lw=1.5)
    ax.text(xs[1], peak * 1.12, label, ha="right", va="bottom", fontsize=8, color=MUTED)
ax.text(0.5, 0.5 * PEAK_BW * 1.4, "DRAM 320 GB/s", rotation=33, fontsize=8, color=MUTED)

for track in ["FP32", "FP16"]:
    t = table[table.track == track]
    ax.scatter(t.AI, t.measured_gflops, s=64, marker=MARKERS[track], color=COLORS[track],
               edgecolors="white", linewidths=1.5, zorder=3, label=f"{track} kernels (measured)")
    # Kernels with the same AI stack vertically; offset labels so they don't collide.
    seen = {}
    for _, r in t.iterrows():
        n = seen.get(r.AI, 0)
        seen[r.AI] = n + 1
        ax.annotate(r.kernel, (r.AI, r.measured_gflops), xytext=(8, -4 + 10 * n),
                    textcoords="offset points", fontsize=7.5, color=INK)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(*xs)
ax.set_ylim(10, 120000)
ax.set_xlabel("Arithmetic intensity (FLOP / byte of global loads)", color=INK)
ax.set_ylabel("GFLOPS", color=INK)
ax.set_title(f"Roofline: SGEMM ladder and WMMA GEMM on T4 ({size}^3)", color=INK, loc="left")
ax.grid(True, which="major", color=GRID, lw=0.8)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color(MUTED)
ax.tick_params(colors=MUTED)
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig("results/roofline.png")
print("saved results/roofline.png")


In [ ]:
%%writefile Makefile
NVCC  ?= nvcc
ARCH  ?= sm_75
FLAGS  = -O3 -std=c++17 -arch=$(ARCH) -lineinfo

all: sgemm hgemm softmax

sgemm: src/sgemm.cu src/kernels.cuh src/common.cuh
	$(NVCC) $(FLAGS) -o $@ src/sgemm.cu -lcublas

hgemm: src/hgemm.cu src/kernels_wmma.cuh src/common.cuh
	$(NVCC) $(FLAGS) -o $@ src/hgemm.cu -lcublas

softmax: src/softmax.cu src/kernels_softmax.cuh src/common.cuh
	$(NVCC) $(FLAGS) -o $@ src/softmax.cu -lcublas

clean:
	rm -f sgemm hgemm softmax


## Build the C++ benchmarks

In [ ]:
!make ARCH=sm_75

## 1. FP32 SGEMM ladder vs cuBLAS
Naive at 4096 takes a few seconds per run; expected.

In [ ]:
!./sgemm 1024 2048 4096

## 2. FP16 tensor-core GEMM (WMMA) vs cuBLAS FP16

In [ ]:
!./hgemm 1024 2048 4096

## 3. Softmax: naive 3-kernel vs fused

In [ ]:
!./softmax

## 4. PyTorch extension: correctness tests + benchmark vs torch
First build of the extension takes 1-2 min.

In [ ]:
!pip install -q ninja

In [ ]:
!python python/test_ops.py

In [ ]:
!python python/bench_torch.py

## 5. Performance model + roofline

In [ ]:
!python python/roofline.py
from IPython.display import Image
Image('results/roofline.png')

## Results summary (latest run)

In [ ]:
import pandas as pd
cols = ["gpu", "kernel", "size", "ms", "perf", "pct_of_cublas", "max_err"]
for f, unit in [("results/sgemm.csv", "GFLOPS"), ("results/hgemm.csv", "TFLOPS")]:
    df = pd.read_csv(f, names=cols).groupby(["kernel", "size"]).tail(1)
    print(f"{f}  (perf in {unit})")
    display(df.pivot(index="kernel", columns="size", values="perf").round(2))
    display(df.pivot(index="kernel", columns="size", values="pct_of_cublas").round(1))
sm = pd.read_csv("results/softmax.csv", names=["gpu","kernel","rows","cols","ms","GBps","speedup","max_err"])
display(sm.groupby(["kernel","rows","cols"]).tail(1))
display(pd.read_csv("results/torch_compare.csv"))

## 6. Profiling
Nsight Compute needs GPU performance counters, which Kaggle often blocks (`ERR_NVGPUCTRPERM`).
If these fail, Nsight Systems (timeline) usually still works; do the `ncu` runs on a rented T4 (AWS g4dn, RunPod, Lambda).

In [ ]:
!which ncu nsys; ls /usr/local/cuda/bin | grep -iE 'ncu|nsys|nsight' || true

In [ ]:
!ncu --kernel-name regex:"sgemm_(smem|regblock|vec4)" --launch-count 3 --section SpeedOfLight --section Occupancy --section WarpStateStats --section MemoryWorkloadAnalysis ./sgemm 2048 2>&1 | tail -120

In [ ]:
!ncu --kernel-name regex:"hgemm_wmma" --launch-count 3 --section SpeedOfLight --section WarpStateStats ./hgemm 2048 2>&1 | tail -80

In [ ]:
!nsys profile -o results/timeline --force-overwrite true ./softmax && nsys stats results/timeline.nsys-rep --report cuda_gpu_kern_sum 2>&1 | tail -25

## Save results
Download `results.zip` from the Output panel and commit `results/` to the repo.

In [ ]:
!zip -qr results.zip results && ls -la results results.zip